# 📉 Embedding Drift Detection & Active Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mattral/production-vlm-engineering/blob/main/notebooks/colab/02_drift_detection_colab.ipynb)

Part of **[production-vlm-engineering](https://github.com/Mattral/production-vlm-engineering)**.

Demonstrates `CosineDriftDetector` (KS-test) and `EWMADriftDetector` (frozen-baseline SPC) plus label-free active-learning triage. Simulates a production stream where distribution shift is injected mid-sequence.

**No GPU required. Runtime: ~1 minute.**

In [1]:
# ── Setup ──────────────────────────────────────────────────────────
import subprocess
import sys

result = subprocess.run(
    ["pip", "install", "git+https://github.com/Mattral/production-vlm-engineering.git",
     "--quiet", "--no-deps"],
    capture_output=True, text=True
)
if result.returncode != 0:
    subprocess.run(["pip", "install", "numpy", "scipy", "matplotlib", "Pillow", "pyyaml",
                    "--quiet"], check=True)
    subprocess.run(["git", "clone", "--depth=1",
                    "https://github.com/Mattral/production-vlm-engineering.git",
                    "/content/pvlm"], capture_output=True)
    sys.path.insert(0, "/content/pvlm/src")
    sys.path.insert(0, "/content/pvlm")

print("✅ Setup complete")

✅ Setup complete


In [2]:
import numpy as np

from production_vlm.drift import CosineDriftDetector, EWMADriftDetector, select_for_active_learning

print("✅ Imports OK")

✅ Imports OK


## Step 1 — Build a reference embedding set

In production this is built from known-good validation data or the first N batches of traffic before any drift is observed. Here we use a synthetic 32-dim Gaussian embedding space.

In [3]:
rng = np.random.default_rng(42)
reference_embeddings = rng.normal(size=(200, 32))

print(f"Reference set: {reference_embeddings.shape[0]} samples, dim={reference_embeddings.shape[1]}")
print("In production: replace with DINOv3 / SigLIP-2 / CLIP embeddings of your validation images")

Reference set: 200 samples, dim=32
In production: replace with DINOv3 / SigLIP-2 / CLIP embeddings of your validation images


## Step 2 — Calibrate both detectors

In [4]:
cosine_detector = CosineDriftDetector(reference_embeddings, alpha=0.01)
ewma_detector   = EWMADriftDetector(lam=0.3, n_sigma=2.0, warmup=5, baseline_n=6)

print(f"KS detector calibrated | reference centroid similarity: "
      f"mean={cosine_detector.reference_similarities.mean():.3f}, "
      f"std={cosine_detector.reference_similarities.std():.3f}")
print("EWMA detector: baseline_n=6 batches, n_sigma=2.0, lam=0.3")

KS detector calibrated | reference centroid similarity: mean=0.762, std=0.101
EWMA detector: baseline_n=6 batches, n_sigma=2.0, lam=0.3


## Step 3 — Simulate a production stream with mid-stream distribution shift

Batches 0-5: in-distribution (same space as reference)  
Batches 6-11: **drifted** (shifted by magnitude=12 in a fixed direction — simulates a camera sensor change, new document format, or style change)

In [5]:
shift_direction = rng.normal(size=32)
shift_direction /= np.linalg.norm(shift_direction)
DRIFT_STARTS_AT = 6

print(f"{'Batch':>5}  {'Drifted':>8}  {'KS stat':>8}  {'p-value':>12}  {'KS flag':>8}  {'EWMA flag':>10}  {'AL queue':>8}")
print("-" * 75)
al_total = 0
for batch_idx in range(12):
    is_drifted = batch_idx >= DRIFT_STARTS_AT
    batch = rng.normal(size=(25, 32))
    if is_drifted:
        batch += 12.0 * shift_direction

    ks  = cosine_detector.score_batch(batch)
    ewma = ewma_detector.update(ks.batch_mean_similarity)
    flagged = ks.is_drift or ewma.is_drift
    al_selected = select_for_active_learning([ks], batch, top_k=5) if flagged else []
    al_total += len(al_selected)

    flag_sym  = "🔴 DRIFT" if ks.is_drift   else "🟢 ok   "
    ewma_sym  = "🔴 DRIFT" if ewma.is_drift else "🟢 ok   "
    print(f"{batch_idx:>5}  {'YES' if is_drifted else 'no':>8}  "
          f"{ks.score:>8.4f}  {ks.p_value:>12.3e}  {flag_sym}  {ewma_sym}  {len(al_selected):>8}")

print(f"\nTotal AL queue: {al_total} samples flagged for human labeling / retraining")

Batch   Drifted   KS stat       p-value   KS flag   EWMA flag  AL queue
---------------------------------------------------------------------------
    0        no    0.2100  2.529e-01  🟢 ok     🟢 ok          0
    1        no    0.1100  7.894e-01  🟢 ok     🟢 ok          0
    2        no    0.1700  4.504e-01  🟢 ok     🟢 ok          0
    3        no    0.1900  3.378e-01  🟢 ok     🟢 ok          0
    4        no    0.1600  5.181e-01  🟢 ok     🟢 ok          0
    5        no    0.1500  5.886e-01  🟢 ok     🟢 ok          0
    6       YES    0.6900  1.244e-10  🔴 DRIFT  🔴 DRIFT       5
    7       YES    0.7000  3.441e-11  🔴 DRIFT  🔴 DRIFT       5
    8       YES    0.6900  1.244e-10  🔴 DRIFT  🟢 ok          5
    9       YES    0.6900  1.244e-10  🔴 DRIFT  🟢 ok          5
   10       YES    0.7000  3.441e-11  🔴 DRIFT  🟢 ok          5
   11       YES    0.6800  2.839e-09  🔴 DRIFT  🟢 ok          5

Total AL queue: 30 samples flagged for human labeling / retraining


## Why EWMA re-centers after the initial onset

The EWMA mean adapts to the new regime within a few batches — so it correctly fires an **onset alert** (batch 6) rather than a persistent alarm. The KS detector stays triggered because it always compares to the *fixed* reference set, not an adaptive baseline. Both behaviors are intentional:

| Detector | Semantic | Use case |
|---|---|---|
| `CosineDriftDetector` | Persistent — fires every batch while drifted | Monitoring dashboards, SLO alerts |
| `EWMADriftDetector` | Onset — fires at the moment of change | PagerDuty / incident alerting |

## ⚠️ The frozen-baseline design was a real bug fix

A naive EWMA that continuously re-estimates variance from the stream is **self-defeating**: the shift itself inflates the variance estimate and widens the control limits just when they need to be tight. This was caught during development — see `production_vlm.drift.EWMADriftDetector` docstring.

## Try it: adjust the drift magnitude

Change `DRIFT_MAGNITUDE` below to see how small a shift the detector can catch:

In [6]:
DRIFT_MAGNITUDE = 6.0  # try 3.0, 6.0, 9.0, 12.0

rng2 = np.random.default_rng(99)
ref2 = rng2.normal(size=(200, 32))
det2 = CosineDriftDetector(ref2, alpha=0.01)
shift2 = rng2.normal(size=32); shift2 /= np.linalg.norm(shift2)

detected = None
for i in range(12):
    batch = rng2.normal(size=(25, 32))
    if i >= 6:
        batch += DRIFT_MAGNITUDE * shift2
    r = det2.score_batch(batch)
    if r.is_drift and i >= 6 and detected is None:
        detected = i

print(f"Drift magnitude={DRIFT_MAGNITUDE}: {'detected at batch ' + str(detected) if detected is not None else 'NOT detected'}")
print("Try magnitude=3.0 to see a failure case — drift monitoring has real sensitivity limits")

Drift magnitude=6.0: detected at batch 6
Try magnitude=3.0 to see a failure case — drift monitoring has real sensitivity limits


---
**Next**: [03 — Robustness & Safety Guard](03_robustness_guard_colab.ipynb)

**Repo**: [github.com/Mattral/production-vlm-engineering](https://github.com/Mattral/production-vlm-engineering)  
**Reference**: Massey (1951) KS test · Montgomery (2020) SPC · Settles (2009) Active Learning survey